# الدرس 03 - أنماط تصميم العملاء الوكلاء

في هذا الدرس، نستكشف ثلاثة أنماط تصميم أساسية لبناء عملاء ذكاء اصطناعي فعّالين:

1. **تعليمات واضحة للعميل** — صياغة مطالبات دقيقة تعرف الدور وتوجه سلوك العميل  
2. **إخراج منظم باستخدام نماذج Pydantic** — ضمان أن يعيد العملاء بيانات متوقعة ومتحقق منها  
3. **عملاء بمسؤولية واحدة** — تصميم عملاء مركزين يقوم كل منهم بشيء واحد بشكل جيد  

سنطبق كل نمط على سيناريو **موفر توصيات وجهات السفر**، نطور تدريجياً نظامًا يمكنه اقتراح وجهات، التحقق من التوافر، والتعامل مع اللوجستيات.


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## النمط 1: تعليمات واضحة للوكيل

النمط الأكثر تأثيرًا هو أيضًا الأبسط: كتابة تعليمات واضحة ومفصلة للوكيل الخاص بك.

تعريف التعليمات الجيدة:
- **من** هو الوكيل (الشخصية والأسلوب)
- **ماذا** يجب عليه أن يفعل (المسؤوليات خطوة بخطوة)
- **كيف** يجب أن يتصرف (القيود والأسلوب)

فيما يلي، ننشئ وكيل كونسيرج سفر مع تعليمات صريحة تشكل كل رد ينتجه.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## النمط 2: الإخراج المنظم مع نماذج Pydantic

النص الحر مفيد للمحادثة، لكن الأنظمة اللاحقة تحتاج إلى بيانات منظمة.  
من خلال إقران **نماذج Pydantic** مع **دالة أداة**، يمكننا:

- تحديد مخطط دقيق لإخراج الوكيل  
- التحقق من صحة الاستجابات تلقائيًا  
- دمج نتائج الوكيل في منطق التطبيق بشكل موثوق  

نقدم أيضًا أداة تعيد تفاصيل الوجهة بحيث يستند الوكيل في توصياته إلى بيانات حقيقية.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## النمط 3: وكلاء المسؤولية الفردية

تستفيد المهام المعقدة من تقسيم العمل عبر عدة وكلاء مركّزين، كل منهم يتحمل مسؤولية واحدة فقط:

- **خبير الوجهة** يعرف عن الأماكن والتوافر
- **مخطط اللوجستيات** يتولى الرحلات الجوية والفنادق وخطط الرحلات

هذا يعكس مبدأ هندسة البرمجيات الخاص بـ *فصل الاهتمامات* — حيث يصبح من الأسهل اختبار وصيانة وتحسين كل وكيل بشكل مستقل.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## الملخص

في هذا الدرس طبقنا ثلاث أنماط تصميم وكيلي على سيناريو توصية السفر:

| النمط | الفكرة الرئيسية | الفائدة |
|---|---|---|
| **تعليمات واضحة** | تحديد الشخصية، المسؤوليات، والقيود مسبقًا | سلوك وكيل متسق ومتوافق مع العلامة التجارية |
| **مخرجات منظمة** | استخدام نماذج Pydantic كصيغة للرد | نتائج مُتحققة وقابلة للقراءة آليًا |
| **مسؤولية واحدة** | إعطاء كل وكيل مهمة واحدة مركزة | أسهل في الاختبار، والصيانة، والتكوين |

تتكون هذه الأنماط بشكل طبيعي — يمكنك دمج التعليمات الواضحة مع المخرجات المنظمة داخل وكيل ذو مسؤولية واحدة لبناء أنظمة قوية وجاهزة للإنتاج.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**إخلاء المسؤولية**:  
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
